# Object Detection on COCO80 with Hugging Face Deformable DETR

This notebook demonstrates how to fine-tune the **Deformable DETR** model from Hugging Face Transformers on the **COCO80** dataset for object detection. It covers:

1. Environment setup & installation
2. Loading and preprocessing the COCO dataset
3. Loading the pretrained Deformable DETR model
4. Training loop with the `Trainer` API
5. Evaluation using COCO metrics
6. Inference on a sample image

> **Note:** This notebook assumes you have a GPU runtime (e.g., Google Colab with T4). Adjust `batch_size`, `learning_rate`, and `num_epochs` based on your hardware.

## 1. Environment Setup

Install the required packages. We use `transformers`, `datasets`, `torch`, `torchvision`, `pycocotools`, and `albumentations` for data augmentation.

In [ ]:
!pip install -q transformers datasets torch torchvision pycocotools albumentations opencv-python-headless

In [ ]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import (
    DeformableDetrForObjectDetection,
    DeformableDetrImageProcessor,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset
import evaluate
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 2. Load COCO Dataset

We use the Hugging Face `datasets` library to load the COCO 2017 dataset. The dataset contains `train`, `validation`, and `test` splits.

In [ ]:
dataset = load_dataset('detection-datasets/coco', split='train')
val_dataset = load_dataset('detection-datasets/coco', split='validation')

print(f'Train samples: {len(dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print('Sample keys:', dataset[0].keys())

In [ ]:
# Inspect a single sample
sample = dataset[0]
print('Image size:', sample['image'].size)
print('Objects:', sample['objects'])
print('Categories:', sample['categories'])

## 3. Preprocessing and Data Augmentation

We define a custom `COCODetectionDataset` class that applies augmentations and prepares inputs for the Deformable DETR model. The `DeformableDetrImageProcessor` handles resizing, normalization, and annotation formatting.

In [ ]:
class COCODetectionDataset(Dataset):
    def __init__(self, hf_dataset, processor, transform=None):
        self.dataset = hf_dataset
        self.processor = processor
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image'].convert('RGB')
        objects = sample['objects']

        # Build COCO-style annotation
        annotations = []
        for i in range(len(objects['category'])):
            x, y, w, h = objects['bbox'][i]
            annotations.append({
                'id': i,
                'category_id': objects['category'][i],
                'bbox': [x, y, w, h],
                'area': w * h,
                'iscrowd': 0,
            })

        # Apply augmentations
        if self.transform:
            # Albumentations expects a different format for bboxes
            bboxes = [a['bbox'] for a in annotations]
            transformed = self.transform(
                image=np.array(image),
                bboxes=bboxes,
                category_ids=[a['category_id'] for a in annotations],
            )
            image = Image.fromarray(transformed['image'])
            annotations = []
            for i, (bbox, cat_id) in enumerate(zip(transformed['bboxes'], transformed['category_ids'])):
                x, y, w, h = bbox
                annotations.append({
                    'id': i,
                    'category_id': cat_id,
                    'bbox': [x, y, w, h],
                    'area': w * h,
                    'iscrowd': 0,
                })

        # Prepare for the processor
        target = {'image_id': idx, 'annotations': annotations}
        encoding = self.processor(images=image, annotations=target, return_tensors='pt')
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        return encoding

In [ ]:
processor = DeformableDetrImageProcessor.from_pretrained('SenseTime/deformable-detr')

train_transform = A.Compose([
    A.Resize(800, 800),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']))

val_transform = A.Compose([
    A.Resize(800, 800),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']))

train_ds = COCODetectionDataset(dataset, processor, transform=train_transform)
val_ds = COCODetectionDataset(val_dataset, processor, transform=val_transform)

print(f'Train dataset size: {len(train_ds)}')
print(f'Val dataset size: {len(val_ds)}')

## 4. Load Pretrained Deformable DETR Model

We load the pretrained `SenseTime/deformable-detr` model, which was trained on COCO. The model has 80 object categories.

In [ ]:
model = DeformableDetrForObjectDetection.from_pretrained('SenseTime/deformable-detr')
model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Freeze the backbone (optional)
for name, param in model.named_parameters():
    if 'backbone' in name:
        param.requires_grad = False

print('Model loaded. Number of parameters:', sum(p.numel() for p in model.parameters()))

## 5. Define Training Arguments and Trainer

We use the Hugging Face `Trainer` to handle the training loop. The `TrainingArguments` configure batch size, learning rate, evaluation strategy, and checkpoint saving.

In [ ]:
training_args = TrainingArguments(
    output_dir='./deformable_detr_coco',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=10,
    learning_rate=1e-4,
    weight_decay=1e-4,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=50,
    remove_unused_columns=False,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print('Trainer initialized.')

## 6. Start Training

This will fine-tune the model on the COCO dataset. Training time depends on your GPU.

In [ ]:
trainer.train()

## 7. Evaluate with COCO Metrics

We use the `pycocotools` library to compute standard COCO metrics (mAP@0.5:0.95, mAP@0.5, etc.).

In [ ]:
def evaluate_coco(model, dataset, processor, device='cuda'):
    model.eval()
    results = []
    for idx in range(len(dataset)):
        sample = dataset.dataset[idx]
        image = sample['image'].convert('RGB')
        inputs = processor(images=image, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        # Post-process
        target_sizes = torch.tensor([image.size[::-1]]).to(device)
        preds = processor.post_process_object_detection(
            outputs, threshold=0.3, target_sizes=target_sizes
        )[0]
        for score, label, box in zip(preds['scores'], preds['labels'], preds['boxes']):
            box = box.cpu().numpy()
            results.append({
                'image_id': idx,
                'category_id': label.item(),
                'bbox': [box[0], box[1], box[2] - box[0], box[3] - box[1]],
                'score': score.item(),
            })
    # Load COCO ground truth
    # Note: In practice, you would need the COCO annotation file.
    # For simplicity, we show the structure here.
    return results

# Example: Run evaluation on a small subset
sample_results = evaluate_coco(model, val_ds, processor)
print(f'Generated {len(sample_results)} predictions.')

## 8. Inference on a Single Image

Finally, we demonstrate how to run inference on a sample image and visualize the predictions.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualize_predictions(image, preds, threshold=0.5):
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)
    for score, label, box in zip(preds['scores'], preds['labels'], preds['boxes']):
        if score < threshold:
            continue
        x1, y1, x2, y2 = box.cpu().numpy()
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1, f'{model.config.id2label[label.item()]}: {score:.2f}',
                bbox=dict(facecolor='yellow', alpha=0.5), fontsize=10)
    plt.axis('off')
    plt.show()

# Load a sample image
sample_idx = 0
sample_image = val_dataset[sample_idx]['image'].convert('RGB')
inputs = processor(images=sample_image, return_tensors='pt').to('cuda' if torch.cuda.is_available() else 'cpu')
with torch.no_grad():
    outputs = model(**inputs)
target_sizes = torch.tensor([sample_image.size[::-1]]).to('cuda' if torch.cuda.is_available() else 'cpu')
preds = processor.post_process_object_detection(outputs, threshold=0.3, target_sizes=target_sizes)[0]
visualize_predictions(sample_image, preds)

## 9. Save the Fine-Tuned Model

Save the model and processor for later use.

In [ ]:
model.save_pretrained('./deformable_detr_coco_finetuned')
processor.save_pretrained('./deformable_detr_coco_finetuned')
print('Model saved to ./deformable_detr_coco_finetuned')

## Summary

This notebook provided a complete pipeline for fine-tuning Deformable DETR on the COCO80 dataset. Key points:

- The `DeformableDetrImageProcessor` handles annotation formatting automatically.
- The `Trainer` API simplifies the training loop.
- Post-processing with `post_process_object_detection` yields bounding boxes and scores.
- COCO evaluation can be performed with `pycocotools`.

For more details, refer to the [Hugging Face Deformable DETR documentation](https://huggingface.co/docs/transformers/model_doc/deformable_detr)[reference:0].